
## Влияние методов балансировки данных на качество моделей ML в задаче обнаружения мошеннических транзакций
 
### Автор: Игорь Алексеев
### Дата: Апрель 2026
 
## Структура:
###   Блок 1 — Импорт библиотек
###   Блок 2 — Загрузка и первичный анализ данных (EDA)
###   Блок 3 — Предобработка данных
###   Блок 4 — Методы балансировки
###   Блок 5 — Обучение моделей и сравнение
###   Блок 6 — Визуализация результатов (таблицы + графики для статьи)



In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("sgpjesus/bank-account-fraud-dataset-neurips-2022")

print("Path to dataset files:", path)
# Download latest version
path = kagglehub.dataset_download("mlg-ulb/creditcardfraud")

print("Path to dataset files:", path)

## БЛОК 1: ИМПОРТ БИБЛИОТЕК

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, 
    roc_auc_score, classification_report, 
    confusion_matrix, precision_recall_curve, auc
)
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from imblearn.pipeline import Pipeline as ImbPipeline
import warnings
warnings.filterwarnings('ignore')

# Настройки визуализации
plt.rcParams['figure.figsize'] = (12, 6)
plt.rcParams['font.size'] = 12
sns.set_style('whitegrid')

print("✅ Все библиотеки загружены")

In [ ]:
df_cc = pd.read_csv('/kaggle/input/datasets/organizations/mlg-ulb/creditcardfraud/creditcard.csv')

print("ДАТАСЕТ 1: Credit Card Fraud (ULB)")
print(f"Размер: {df_cc.shape}")
print(f"\nРаспределение классов:")
print(df_cc['Class'].value_counts())
print(f"\nДоля фрода: {df_cc['Class'].mean():.4%}")
print(f"\nПервые строки:")
df_cc.head()


## БЛОК 2.1: ЗАГРУЗКА ДАТАСЕТА 2
### Датасет 2: Bank Account Fraud (NeurIPS 2022) 
### Берём Base-вариант. 1M записей, дисбаланс мягче (~6%)
### Фичи интерпретируемые: income, age, credit_risk_score и др.

In [ ]:
df_baf = pd.read_csv(
    '/kaggle/input/datasets/sgpjesus/bank-account-fraud-dataset-neurips-2022/Base.csv'
)

print("ДАТАСЕТ 2: Bank Account Fraud (NeurIPS 2022)")
print(f"Размер: {df_baf.shape}")
print(f"\nРаспределение классов:")
print(df_baf['fraud_bool'].value_counts())
print(f"\nДоля фрода: {df_baf['fraud_bool'].mean():.4%}")
print(f"\nПервые строки:")
df_baf.head()

## БЛОК 2.2: ВИЗУАЛИЗАЦИЯ ДИСБАЛАНСА
### (Это график в статью хочу) — показывает разницу в дисбалансе между датасетами

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Датасет 1
counts_cc = df_cc['Class'].value_counts()
axes[0].bar(['Легитимные', 'Мошеннические'], counts_cc.values, 
            color=['#2196F3', '#F44336'])
axes[0].set_title(f'Credit Card Fraud\nФрод: {df_cc["Class"].mean():.2%}')
axes[0].set_ylabel('Количество транзакций')
for i, v in enumerate(counts_cc.values):
    axes[0].text(i, v + 1000, str(v), ha='center', fontweight='bold')

# Датасет 2
counts_baf = df_baf['fraud_bool'].value_counts()
axes[1].bar(['Легитимные', 'Мошеннические'], counts_baf.values, 
            color=['#2196F3', '#F44336'])
axes[1].set_title(f'Bank Account Fraud\nФрод: {df_baf["fraud_bool"].mean():.2%}')
axes[1].set_ylabel('Количество транзакций')
for i, v in enumerate(counts_baf.values):
    axes[1].text(i, v + 5000, str(v), ha='center', fontweight='bold')

plt.suptitle('Сравнение дисбаланса классов в двух датасетах', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('class_imbalance_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

## БЛОК 3: ПРЕДОБРАБОТКА

### Датасет 1: Credit Card
### Уже почти чистый (PCA-фичи), нужно только масштабировать Amount и Time


In [ ]:
X_cc = df_cc.drop('Class', axis=1).copy()
y_cc = df_cc['Class'].copy()

# Масштабируем Amount и Time (V1-V28 уже масштабированы после PCA)
scaler = StandardScaler()
X_cc[['Amount', 'Time']] = scaler.fit_transform(X_cc[['Amount', 'Time']])

print(f"Credit Card — X: {X_cc.shape}, y: {y_cc.shape}")
print(f"Пропуски: {X_cc.isnull().sum().sum()}")


# --- Датасет 2: Bank Account Fraud ---
# Нужно обработать категориальные фичи и пропуски

X_baf = df_baf.drop('fraud_bool', axis=1).copy()
y_baf = df_baf['fraud_bool'].copy()

# Смотрим типы данных
print(f"\nBank Account Fraud — X: {X_baf.shape}, y: {y_baf.shape}")
print(f"Типы данных:\n{X_baf.dtypes.value_counts()}")

# Кодируем категориальные переменные (One-Hot Encoding)
cat_cols = X_baf.select_dtypes(include=['object', 'bool']).columns.tolist()
print(f"\nКатегориальные колонки: {cat_cols}")

X_baf = pd.get_dummies(X_baf, columns=cat_cols, drop_first=True)

# Заполняем пропуски медианой (если есть)
X_baf = X_baf.fillna(X_baf.median())

# Масштабируем числовые фичи
scaler_baf = StandardScaler()
X_baf = pd.DataFrame(
    scaler_baf.fit_transform(X_baf), 
    columns=X_baf.columns
)

print(f"\nBank Account Fraud после обработки — X: {X_baf.shape}")
print(f"Пропуски: {X_baf.isnull().sum().sum()}")


## БЛОК 4: ЭКСПЕРИМЕНТ 

> 3 модели: LogReg, Random Forest, XGBoost
> 
> 3 режима балансировки: без балансировки, SMOTE, Random Undersampling
> 
> На 2 датасетах с разным уровнем дисбаланса
>
> Используем Stratified 5-Fold CV для надёжности результатов



In [ ]:
def run_experiment(X, y, dataset_name, n_splits=5):
    """
    Запускает эксперимент: 3 модели × 3 метода балансировки.
    Возвращает DataFrame с результатами.
    """
    
    # Определяем модели
    models = {
        'Logistic Regression': LogisticRegression(max_iter=1000, random_state=42),
        'Random Forest': RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1),
        'XGBoost': XGBClassifier(n_estimators=100, random_state=42, 
                                  use_label_encoder=False, eval_metric='logloss')
    }
    
    # Определяем методы балансировки
    balancing_methods = {
        'Без балансировки': None,
        'SMOTE': SMOTE(random_state=42),
        'Random Undersampling': RandomUnderSampler(random_state=42),
        'Class Weight Balanced': 'balanced' 
    }
    
    results = []
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=42)
    
    total_experiments = len(models) * len(balancing_methods)
    current = 0
    
    for model_name, model in models.items():
        for balance_name, balancer in balancing_methods.items():
            current += 1
            print(f"  [{current}/{total_experiments}] {model_name} + {balance_name}...", end=" ")
            
            fold_metrics = {
                'f1': [], 'precision': [], 'recall': [], 
                'roc_auc': [], 'pr_auc': []
            }
            
            for fold, (train_idx, test_idx) in enumerate(skf.split(X, y)):
                X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
                y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
                
                # Применяем балансировку (только к train!)
                if balancer is not None and balance_name != 'Class Weight Balanced':
                    X_train_bal, y_train_bal = balancer.fit_resample(X_train, y_train)
                else:
                    X_train_bal, y_train_bal = X_train, y_train
                
                # Обучаем модель
                if balance_name == 'Class Weight Balanced':
                    # Передаём class_weight прямо в модель
                    params = model.get_params()
                    if hasattr(model, 'class_weight'):
                        params['class_weight'] = 'balanced'
                    elif hasattr(model, 'scale_pos_weight'):  # XGBoost
                        neg = sum(y_train_bal == 0)
                        pos = sum(y_train_bal == 1)
                        params['scale_pos_weight'] = neg / pos
                    clf = model.__class__(**params)
                else:
                    clf = model.__class__(**model.get_params())
                
                clf.fit(X_train_bal, y_train_bal)
                                
                # Предсказания
                y_pred = clf.predict(X_test)
                y_proba = clf.predict_proba(X_test)[:, 1]
                
                # Метрики
                fold_metrics['f1'].append(f1_score(y_test, y_pred))
                fold_metrics['precision'].append(precision_score(y_test, y_pred))
                fold_metrics['recall'].append(recall_score(y_test, y_pred))
                fold_metrics['roc_auc'].append(roc_auc_score(y_test, y_proba))
                
                # PR-AUC (важнее ROC-AUC при дисбалансе!)
                prec_curve, rec_curve, _ = precision_recall_curve(y_test, y_proba)
                fold_metrics['pr_auc'].append(auc(rec_curve, prec_curve))
            
            # Средние метрики по фолдам
            result = {
                'Dataset': dataset_name,
                'Model': model_name,
                'Balancing': balance_name,
                'F1': np.mean(fold_metrics['f1']),
                'F1_std': np.std(fold_metrics['f1']),
                'Precision': np.mean(fold_metrics['precision']),
                'Recall': np.mean(fold_metrics['recall']),
                'ROC-AUC': np.mean(fold_metrics['roc_auc']),
                'PR-AUC': np.mean(fold_metrics['pr_auc']),
            }
            results.append(result)
            print(f"F1={result['F1']:.4f}, ROC-AUC={result['ROC-AUC']:.4f}")
    
    return pd.DataFrame(results)


# Запуск экспериментов

# Датасет 1: Credit Card 
print("ЭКСПЕРИМЕНТ 1: Credit Card Fraud Detection")
results_cc = run_experiment(X_cc, y_cc, 'Credit Card Fraud')

# Датасет 2: Bank Account Fraud
print("ЭКСПЕРИМЕНТ 2: Bank Account Fraud (NeurIPS 2022)")

# Берём сэмпл 200k для скорости (в статье упомянем это)
sample_size = 200000
idx = X_baf.sample(sample_size, random_state=42).index
X_baf_sample = X_baf.loc[idx].reset_index(drop=True)
y_baf_sample = y_baf.loc[idx].reset_index(drop=True)
print(f"Используем сэмпл: {sample_size} записей")

results_baf = run_experiment(X_baf_sample, y_baf_sample, 'Bank Account Fraud')

# Объединяем результаты
all_results = pd.concat([results_cc, results_baf], ignore_index=True)
print("\n✅ Все эксперименты завершены!")

## БЛОК 5: ТАБЛИЦА РЕЗУЛЬТАТОВ

In [ ]:
print("СВОДНАЯ ТАБЛИЦА РЕЗУЛЬТАТОВ")

# Форматируем для красивого вывода
display_cols = ['Dataset', 'Model', 'Balancing', 'F1', 'Precision', 'Recall', 'ROC-AUC', 'PR-AUC']
results_display = all_results[display_cols].copy()

# Округляем
for col in ['F1', 'Precision', 'Recall', 'ROC-AUC', 'PR-AUC']:
    results_display[col] = results_display[col].round(4)

print("\n--- Credit Card Fraud ---")
print(results_display[results_display['Dataset'] == 'Credit Card Fraud'].to_string(index=False))

print("\n--- Bank Account Fraud ---")
print(results_display[results_display['Dataset'] == 'Bank Account Fraud'].to_string(index=False))



## БЛОК 6: ВИЗУАЛИЗАЦИИ ДЛЯ СТАТЬИ

In [ ]:
# --- График 1: Сравнение F1-score по моделям и методам балансировки ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, dataset in enumerate(['Credit Card Fraud', 'Bank Account Fraud']):
    df_plot = all_results[all_results['Dataset'] == dataset]
    
    pivot = df_plot.pivot(index='Model', columns='Balancing', values='F1')
    pivot = pivot[['Без балансировки', 'SMOTE', 'Random Undersampling']]
    
    pivot.plot(kind='bar', ax=axes[i], rot=15, 
               color=['#78909C', '#2196F3', '#FF9800'])
    axes[i].set_title(f'{dataset}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('F1-Score')
    axes[i].set_ylim(0, 1)
    axes[i].legend(title='Метод балансировки', fontsize=9)
    axes[i].set_xlabel('')

plt.suptitle('Влияние методов балансировки на F1-Score', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('f1_comparison.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# --- График 2: Сравнение ROC-AUC ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, dataset in enumerate(['Credit Card Fraud', 'Bank Account Fraud']):
    df_plot = all_results[all_results['Dataset'] == dataset]
    
    pivot = df_plot.pivot(index='Model', columns='Balancing', values='ROC-AUC')
    pivot = pivot[['Без балансировки', 'SMOTE', 'Random Undersampling']]
    
    pivot.plot(kind='bar', ax=axes[i], rot=15, 
               color=['#78909C', '#2196F3', '#FF9800'])
    axes[i].set_title(f'{dataset}', fontsize=13, fontweight='bold')
    axes[i].set_ylabel('ROC-AUC')
    axes[i].set_ylim(0.5, 1)
    axes[i].legend(title='Метод балансировки', fontsize=9)
    axes[i].set_xlabel('')

plt.suptitle('Влияние методов балансировки на ROC-AUC', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('roc_auc_comparison.png', dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# --- График 3: Precision vs Recall (trade-off) ---
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

for i, dataset in enumerate(['Credit Card Fraud', 'Bank Account Fraud']):
    df_plot = all_results[all_results['Dataset'] == dataset]
    
    colors = {'Без балансировки': '#78909C', 'SMOTE': '#2196F3', 
              'Random Undersampling': '#FF9800', 'Class Weight Balanced': '#4CAF50'}
    markers = {'Logistic Regression': 'o', 'Random Forest': 's', 'XGBoost': 'D'}
    
    for _, row in df_plot.iterrows():
        axes[i].scatter(
            row['Recall'], row['Precision'],
            c=colors[row['Balancing']], 
            marker=markers[row['Model']],
            s=150, edgecolors='black', linewidth=0.5,
            label=f"{row['Model']} + {row['Balancing']}"
        )
    
    axes[i].set_title(f'{dataset}', fontsize=13, fontweight='bold')
    axes[i].set_xlabel('Recall')
    axes[i].set_ylabel('Precision')
    axes[i].set_xlim(0, 1.05)
    axes[i].set_ylim(0, 1.05)
    
    # Убираем дублирующиеся легенды
    handles, labels = axes[i].get_legend_handles_labels()
    by_label = dict(zip(labels, handles))
    if i == 1:
        axes[i].legend(by_label.values(), by_label.keys(), 
                      fontsize=7, loc='lower left')

plt.suptitle('Precision vs Recall: влияние балансировки', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('precision_recall_scatter.png', dpi=150, bbox_inches='tight')
plt.show()



In [ ]:
# --- График 4: Тепловая карта (Heatmap) — для наглядности в статье ---
fig, axes = plt.subplots(1, 2, figsize=(14, 6))

for i, dataset in enumerate(['Credit Card Fraud', 'Bank Account Fraud']):
    df_plot = all_results[all_results['Dataset'] == dataset]
    
    # Создаём мульти-индекс для красивой таблицы
    df_plot['Label'] = df_plot['Model'] + '\n' + df_plot['Balancing']
    
    heatmap_data = df_plot.set_index('Label')[['F1', 'Precision', 'Recall', 'ROC-AUC', 'PR-AUC']]
    
    sns.heatmap(heatmap_data, annot=True, fmt='.3f', cmap='RdYlGn',
                ax=axes[i], vmin=0, vmax=1, linewidths=0.5)
    axes[i].set_title(f'{dataset}', fontsize=12, fontweight='bold')
    axes[i].set_ylabel('')

plt.suptitle('Сводная карта метрик по всем экспериментам', 
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.savefig('heatmap_results.png', dpi=150, bbox_inches='tight')
plt.show()

## БЛОК 7: ВЫВОДЫ


In [ ]:
print("КЛЮЧЕВЫЕ ВЫВОДЫ ДЛЯ СТАТЬИ")

# Автоматически находим лучшие комбинации
for dataset in ['Credit Card Fraud', 'Bank Account Fraud']:
    df_d = all_results[all_results['Dataset'] == dataset]
    best_f1 = df_d.loc[df_d['F1'].idxmax()]
    best_auc = df_d.loc[df_d['ROC-AUC'].idxmax()]
    
    print(f"\n📊 {dataset}:")
    print(f"  Лучший F1:      {best_f1['Model']} + {best_f1['Balancing']} → F1={best_f1['F1']:.4f}")
    print(f"  Лучший ROC-AUC: {best_auc['Model']} + {best_auc['Balancing']} → AUC={best_auc['ROC-AUC']:.4f}")
    
    # Сравним: помог ли SMOTE по сравнению с "без балансировки"?
    for model_name in ['Logistic Regression', 'Random Forest', 'XGBoost']:
        f1_none = df_d[(df_d['Model'] == model_name) & 
                       (df_d['Balancing'] == 'Без балансировки')]['F1'].values[0]
        f1_smote = df_d[(df_d['Model'] == model_name) & 
                        (df_d['Balancing'] == 'SMOTE')]['F1'].values[0]
        delta = f1_smote - f1_none
        direction = "↑" if delta > 0 else "↓"
        print(f"  {model_name}: SMOTE {direction} {abs(delta):.4f} от baseline")

print("Графики сохранены как PNG — используй их в статье!")

## БЛОК 8: ЭКСПОРТ ТАБЛИЦЫ В CSV 

In [ ]:
all_results.to_csv('experiment_results.csv', index=False)
print("📁 Результаты сохранены в experiment_results.csv")